In [0]:
%sql
-- ============================================================
-- NOTEBOOK 03
-- CAPA GOLD (Analytics / BI)
--
-- Aquí creamos tablas agregadas para análisis.
-- Estas tablas normalmente alimentan dashboards de Power BI.
--
-- Fuente: silver_schema.superstore_silver
-- Destino: gold_schema
-- ============================================================

USE CATALOG asbdevsuperstore;
USE SCHEMA gold_schema;

-- ============================================================
-- 1) TABLA GOLD: VENTAS POR REGION Y MES
--
-- Uso típico:
-- Dashboard de ventas por región a lo largo del tiempo
-- ============================================================

CREATE TABLE IF NOT EXISTS asbdevsuperstore.gold_schema.sales_by_region_month
USING DELTA
LOCATION 'abfss://superstoredata@adsldevsuperstore.dfs.core.windows.net/gold/sales_by_region_month'
AS
SELECT
  region,
  date_trunc('month', order_date) AS order_month,
  COUNT(DISTINCT order_id) AS orders,
  SUM(quantity) AS units,
  SUM(sales) AS sales,
  SUM(profit) AS profit,
  AVG(discount) AS avg_discount
FROM asbdevsuperstore.silver_schema.superstore_silver
GROUP BY region, date_trunc('month', order_date);

-- Optimiza archivos (mejora performance)
OPTIMIZE asbdevsuperstore.gold_schema.sales_by_region_month;



-- ============================================================
-- 2) TABLA GOLD: VENTAS POR CATEGORIA Y MES
--
-- Uso típico:
-- Analizar performance de categorías de producto
-- (Technology / Furniture / Office Supplies)
-- ============================================================

CREATE TABLE IF NOT EXISTS asbdevsuperstore.gold_schema.sales_by_category_month
USING DELTA
LOCATION 'abfss://superstoredata@adsldevsuperstore.dfs.core.windows.net/gold/sales_by_category_month'
AS
SELECT
  category,
  date_trunc('month', order_date) AS order_month,
  COUNT(DISTINCT order_id) AS orders,
  SUM(quantity) AS units,
  SUM(sales) AS sales,
  SUM(profit) AS profit
FROM asbdevsuperstore.silver_schema.superstore_silver
GROUP BY category, date_trunc('month', order_date);

OPTIMIZE asbdevsuperstore.gold_schema.sales_by_category_month;



-- ============================================================
-- 3) TABLA GOLD: TOP PRODUCTOS POR PROFIT
--
-- Uso típico:
-- identificar los productos más rentables
-- ============================================================

CREATE TABLE IF NOT EXISTS asbdevsuperstore.gold_schema.top_products_profit
USING DELTA
LOCATION 'abfss://superstoredata@adsldevsuperstore.dfs.core.windows.net/gold/top_products_profit'
AS
SELECT
  product_name,
  category,
  SUM(quantity) AS units_sold,
  SUM(sales) AS total_sales,
  SUM(profit) AS total_profit
FROM asbdevsuperstore.silver_schema.superstore_silver
GROUP BY product_name, category
ORDER BY total_profit DESC;

OPTIMIZE asbdevsuperstore.gold_schema.top_products_profit;



-- ============================================================
-- 4) TABLA GOLD: VENTAS POR SEGMENTO DE CLIENTE
--
-- Segmentos típicos:
-- Consumer
-- Corporate
-- Home Office
--
-- Uso típico:
-- analizar qué tipo de cliente genera más ventas
-- ============================================================

CREATE TABLE IF NOT EXISTS asbdevsuperstore.gold_schema.sales_by_segment
USING DELTA
LOCATION 'abfss://superstoredata@adsldevsuperstore.dfs.core.windows.net/gold/sales_by_segment'
AS
SELECT
  segment,
  COUNT(DISTINCT order_id) AS orders,
  SUM(quantity) AS units,
  SUM(sales) AS sales,
  SUM(profit) AS profit,
  AVG(discount) AS avg_discount
FROM asbdevsuperstore.silver_schema.superstore_silver
GROUP BY segment;

OPTIMIZE asbdevsuperstore.gold_schema.sales_by_segment;



-- ============================================================
-- VALIDACION RAPIDA
-- ============================================================

SELECT * 
FROM asbdevsuperstore.gold_schema.sales_by_region_month
LIMIT 10;

SELECT *
FROM asbdevsuperstore.gold_schema.sales_by_category_month
LIMIT 10;

SELECT *
FROM asbdevsuperstore.gold_schema.top_products_profit
LIMIT 10;

SELECT *
FROM asbdevsuperstore.gold_schema.sales_by_segment
LIMIT 10;